# FinRL Hyperparameter Optimization with Walk-Forward Validation

**Purpose**: This notebook performs hyperparameter optimization (HPO) using walk-forward validation to find robust configurations for the PPO trading agent.

**Output**: Best hyperparameters to copy into the v2 trading notebook for deployment.

---

*Disclaimer: Nothing herein is financial advice, and NOT a recommendation to trade real money.*

<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/3-Practical/FinRL_PaperTrading_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Part 1: Install FinRL

In [ ]:
## install finrl library
#!pip install wrds
#!pip install swig
#!pip install -q condacolab
#import condacolab
#condacolab.install()
#!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
#!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

## Import related modules

In [1]:
from finrl.config_tickers import DOW_30_TICKER
from finrl.config import INDICATORS
from finrl.meta.env_stock_trading.env_stocktrading_np import StockTradingEnv
from finrl.meta.env_stock_trading.env_stock_papertrading import AlpacaPaperTrading
from finrl.meta.data_processor import DataProcessor
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline

import numpy as np
import pandas as pd

/opt/finrl/.venv/lib/python3.12/site-packages/pyfolio/pos.py:25: UserWarning: Module "zipline.assets" not found; multipliers will not be applied to position notionals.
  warnings.warn(


## PPO

In [2]:
import os
import time
import gymnasium as gym
import numpy as np
import numpy.random as rd
import torch
import torch.nn as nn
from copy import deepcopy
from torch import Tensor
from torch.distributions.normal import Normal


class ActorPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, action_dim])
        self.action_std_log = nn.Parameter(torch.zeros((1, action_dim)), requires_grad=True)  # trainable parameter

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state).tanh()  # action.tanh()

    def get_action(self, state: Tensor) -> (Tensor, Tensor):  # for exploration
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        action = dist.sample()
        logprob = dist.log_prob(action).sum(1)
        return action, logprob

    def get_logprob_entropy(self, state: Tensor, action: Tensor) -> (Tensor, Tensor):
        action_avg = self.net(state)
        action_std = self.action_std_log.exp()

        dist = Normal(action_avg, action_std)
        logprob = dist.log_prob(action).sum(1)
        entropy = dist.entropy().sum(1)
        return logprob, entropy

    @staticmethod
    def convert_action_for_env(action: Tensor) -> Tensor:
        return action.tanh()


class CriticPPO(nn.Module):
    def __init__(self, dims: [int], state_dim: int, _action_dim: int):
        super().__init__()
        self.net = build_mlp(dims=[state_dim, *dims, 1])

    def forward(self, state: Tensor) -> Tensor:
        return self.net(state)  # advantage value


def build_mlp(dims: [int]) -> nn.Sequential:  # MLP (MultiLayer Perceptron)
    net_list = []
    for i in range(len(dims) - 1):
        net_list.extend([nn.Linear(dims[i], dims[i + 1]), nn.ReLU()])
    del net_list[-1]  # remove the activation of output layer
    return nn.Sequential(*net_list)


class Config:
    def __init__(self, agent_class=None, env_class=None, env_args=None):
        self.env_class = env_class  # env = env_class(**env_args)
        self.env_args = env_args  # env = env_class(**env_args)

        if env_args is None:  # dummy env_args
            env_args = {'env_name': None, 'state_dim': None, 'action_dim': None, 'if_discrete': None}
        self.env_name = env_args['env_name']  # the name of environment. Be used to set 'cwd'.
        self.state_dim = env_args['state_dim']  # vector dimension (feature number) of state
        self.action_dim = env_args['action_dim']  # vector dimension (feature number) of action
        self.if_discrete = env_args['if_discrete']  # discrete or continuous action space

        self.agent_class = agent_class  # agent = agent_class(...)

        '''Arguments for reward shaping'''
        self.gamma = 0.99  # discount factor of future rewards
        self.reward_scale = 1.0  # an approximate target reward usually be closed to 256

        '''Arguments for training'''
        self.gpu_id = -1  # `int` means the ID of single GPU, -1 means CPU (CHANGED TO FORCE CPU)
        self.net_dims = (64, 32)  # the middle layer dimension of MLP (MultiLayer Perceptron)
        self.learning_rate = 6e-5  # 2 ** -14 ~= 6e-5
        self.soft_update_tau = 5e-3  # 2 ** -8 ~= 5e-3
        self.batch_size = int(128)  # num of transitions sampled from replay buffer.
        self.horizon_len = int(2000)  # collect horizon_len step while exploring, then update network
        self.buffer_size = None  # ReplayBuffer size. Empty the ReplayBuffer for on-policy.
        self.repeat_times = 8.0  # repeatedly update network using ReplayBuffer to keep critic's loss small

        '''Arguments for evaluate'''
        self.cwd = None  # current working directory to save model. None means set automatically
        self.break_step = +np.inf  # break training if 'total_step > break_step'
        self.eval_times = int(32)  # number of times that get episodic cumulative return
        self.eval_per_step = int(2e4)  # evaluate the agent per training steps

    def init_before_training(self):
        if self.cwd is None:  # set cwd (current working directory) for saving model
            self.cwd = f'./{self.env_name}_{self.agent_class.__name__[5:]}'
        os.makedirs(self.cwd, exist_ok=True)


def get_gym_env_args(env, if_print: bool) -> dict:
    if {'unwrapped', 'observation_space', 'action_space', 'spec'}.issubset(dir(env)):  # isinstance(env, gym.Env):
        env_name = env.unwrapped.spec.id
        state_shape = env.observation_space.shape
        state_dim = state_shape[0] if len(state_shape) == 1 else state_shape  # sometimes state_dim is a list

        if_discrete = isinstance(env.action_space, gym.spaces.Discrete)
        if if_discrete:  # make sure it is discrete action space
            action_dim = env.action_space.n
        elif isinstance(env.action_space, gym.spaces.Box):  # make sure it is continuous action space
            action_dim = env.action_space.shape[0]

    env_args = {'env_name': env_name, 'state_dim': state_dim, 'action_dim': action_dim, 'if_discrete': if_discrete}
    print(f"env_args = {repr(env_args)}") if if_print else None
    return env_args


def kwargs_filter(function, kwargs: dict) -> dict:
    import inspect
    sign = inspect.signature(function).parameters.values()
    sign = {val.name for val in sign}
    common_args = sign.intersection(kwargs.keys())
    return {key: kwargs[key] for key in common_args}  # filtered kwargs


def build_env(env_class=None, env_args=None):
    if env_class.__module__ == 'gym.envs.registration':  # special rule
        env = env_class(id=env_args['env_name'])
    else:
        env = env_class(**kwargs_filter(env_class.__init__, env_args.copy()))
    for attr_str in ('env_name', 'state_dim', 'action_dim', 'if_discrete'):
        setattr(env, attr_str, env_args[attr_str])
    return env


class AgentBase:
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.state_dim = state_dim
        self.action_dim = action_dim

        self.gamma = args.gamma
        self.batch_size = args.batch_size
        self.repeat_times = args.repeat_times
        self.reward_scale = args.reward_scale
        self.soft_update_tau = args.soft_update_tau

        self.states = None  # assert self.states == (1, state_dim)
        self.device = torch.device(f"cuda:{gpu_id}" if (torch.cuda.is_available() and (gpu_id >= 0)) else "cpu")

        act_class = getattr(self, "act_class", None)
        cri_class = getattr(self, "cri_class", None)
        self.act = self.act_target = act_class(net_dims, state_dim, action_dim).to(self.device)
        self.cri = self.cri_target = cri_class(net_dims, state_dim, action_dim).to(self.device) \
            if cri_class else self.act

        self.act_optimizer = torch.optim.Adam(self.act.parameters(), args.learning_rate)
        self.cri_optimizer = torch.optim.Adam(self.cri.parameters(), args.learning_rate) \
            if cri_class else self.act_optimizer

        self.criterion = torch.nn.SmoothL1Loss()

    @staticmethod
    def optimizer_update(optimizer, objective: Tensor):
        optimizer.zero_grad()
        objective.backward()
        optimizer.step()

    @staticmethod
    def soft_update(target_net: torch.nn.Module, current_net: torch.nn.Module, tau: float):
        for tar, cur in zip(target_net.parameters(), current_net.parameters()):
            tar.data.copy_(cur.data * tau + tar.data * (1.0 - tau))


class AgentPPO(AgentBase):
    def __init__(self, net_dims: [int], state_dim: int, action_dim: int, gpu_id: int = 0, args: Config = Config()):
        self.if_off_policy = False
        self.act_class = getattr(self, "act_class", ActorPPO)
        self.cri_class = getattr(self, "cri_class", CriticPPO)
        AgentBase.__init__(self, net_dims, state_dim, action_dim, gpu_id, args)

        self.ratio_clip = getattr(args, "ratio_clip", 0.25)  # `ratio.clamp(1 - clip, 1 + clip)`
        self.lambda_gae_adv = getattr(args, "lambda_gae_adv", 0.95)  # could be 0.80~0.99
        self.lambda_entropy = getattr(args, "lambda_entropy", 0.01)  # could be 0.00~0.10
        self.lambda_entropy = torch.tensor(self.lambda_entropy, dtype=torch.float32, device=self.device)

    def explore_env(self, env, horizon_len: int) -> [Tensor]:
        states = torch.zeros((horizon_len, self.state_dim), dtype=torch.float32).to(self.device)
        actions = torch.zeros((horizon_len, self.action_dim), dtype=torch.float32).to(self.device)
        logprobs = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        rewards = torch.zeros(horizon_len, dtype=torch.float32).to(self.device)
        dones = torch.zeros(horizon_len, dtype=torch.bool).to(self.device)

        ary_state = self.states[0]

        get_action = self.act.get_action
        convert = self.act.convert_action_for_env
        for i in range(horizon_len):
            state = torch.as_tensor(ary_state, dtype=torch.float32, device=self.device)
            action, logprob = [t.squeeze(0) for t in get_action(state.unsqueeze(0))[:2]]

            ary_action = convert(action).detach().cpu().numpy()
            ary_state, reward, done, _, _ = env.step(ary_action)
            if done:
                ary_state, _ = env.reset()

            states[i] = state
            actions[i] = action
            logprobs[i] = logprob
            rewards[i] = reward
            dones[i] = done

        self.states[0] = ary_state
        rewards = (rewards * self.reward_scale).unsqueeze(1)
        undones = (1 - dones.type(torch.float32)).unsqueeze(1)
        return states, actions, logprobs, rewards, undones

    def update_net(self, buffer) -> [float]:
        with torch.no_grad():
            states, actions, logprobs, rewards, undones = buffer
            buffer_size = states.shape[0]

            '''get advantages reward_sums'''
            bs = 2 ** 10  # set a smaller 'batch_size' when out of GPU memory.
            values = [self.cri(states[i:i + bs]) for i in range(0, buffer_size, bs)]
            values = torch.cat(values, dim=0).squeeze(1)  # values.shape == (buffer_size, )

            advantages = self.get_advantages(rewards, undones, values)  # advantages.shape == (buffer_size, )
            reward_sums = advantages + values  # reward_sums.shape == (buffer_size, )
            del rewards, undones, values

            advantages = (advantages - advantages.mean()) / (advantages.std(dim=0) + 1e-5)
        assert logprobs.shape == advantages.shape == reward_sums.shape == (buffer_size,)

        '''update network'''
        obj_critics = 0.0
        obj_actors = 0.0

        update_times = int(buffer_size * self.repeat_times / self.batch_size)
        assert update_times >= 1
        for _ in range(update_times):
            indices = torch.randint(buffer_size, size=(self.batch_size,), requires_grad=False)
            state = states[indices]
            action = actions[indices]
            logprob = logprobs[indices]
            advantage = advantages[indices]
            reward_sum = reward_sums[indices]

            value = self.cri(state).squeeze(1)  # critic network predicts the reward_sum (Q value) of state
            obj_critic = self.criterion(value, reward_sum)
            self.optimizer_update(self.cri_optimizer, obj_critic)

            new_logprob, obj_entropy = self.act.get_logprob_entropy(state, action)
            ratio = (new_logprob - logprob.detach()).exp()
            surrogate1 = advantage * ratio
            surrogate2 = advantage * ratio.clamp(1 - self.ratio_clip, 1 + self.ratio_clip)
            obj_surrogate = torch.min(surrogate1, surrogate2).mean()

            obj_actor = obj_surrogate + obj_entropy.mean() * self.lambda_entropy
            self.optimizer_update(self.act_optimizer, -obj_actor)

            obj_critics += obj_critic.item()
            obj_actors += obj_actor.item()
        a_std_log = getattr(self.act, 'a_std_log', torch.zeros(1)).mean()
        return obj_critics / update_times, obj_actors / update_times, a_std_log.item()

    def get_advantages(self, rewards: Tensor, undones: Tensor, values: Tensor) -> Tensor:
        advantages = torch.empty_like(values)  # advantage value

        masks = undones * self.gamma
        horizon_len = rewards.shape[0]

        next_state = torch.tensor(self.states, dtype=torch.float32).to(self.device)
        next_value = self.cri(next_state).detach()[0, 0]

        advantage = 0  # last_gae_lambda
        for t in range(horizon_len - 1, -1, -1):
            delta = rewards[t] + masks[t] * next_value - values[t]
            advantages[t] = advantage = delta + masks[t] * self.lambda_gae_adv * advantage
            next_value = values[t]
        return advantages


class PendulumEnv(gym.Wrapper):  # a demo of custom gym env
    def __init__(self):
        gym.logger.set_level(40)  # Block warning
        gym_env_name = "Pendulum-v0" if gym.__version__ < '0.18.0' else "Pendulum-v1"
        super().__init__(env=gym.make(gym_env_name))

        '''the necessary env information when you design a custom env'''
        self.env_name = gym_env_name  # the name of this env.
        self.state_dim = self.observation_space.shape[0]  # feature number of state
        self.action_dim = self.action_space.shape[0]  # feature number of action
        self.if_discrete = False  # discrete action or continuous action

    def reset(self) -> np.ndarray:  # reset the agent in env
        resetted_env, _ = self.env.reset()
        return resetted_env

    def step(self, action: np.ndarray) -> (np.ndarray, float, bool, dict):  # agent interacts in env
        # We suggest that adjust action space to (-1, +1) when designing a custom env.
        state, reward, done, info_dict, _ = self.env.step(action * 2)
        return state.reshape(self.state_dim), float(reward), done, info_dict


def train_agent(args: Config):
    args.init_before_training()

    env = build_env(args.env_class, args.env_args)
    agent = args.agent_class(args.net_dims, args.state_dim, args.action_dim, gpu_id=args.gpu_id, args=args)

    new_env, _ = env.reset()
    agent.states = new_env[np.newaxis, :]

    evaluator = Evaluator(eval_env=build_env(args.env_class, args.env_args),
                          eval_per_step=args.eval_per_step,
                          eval_times=args.eval_times,
                          cwd=args.cwd)
    torch.set_grad_enabled(False)
    while True: # start training
        buffer_items = agent.explore_env(env, args.horizon_len)

        torch.set_grad_enabled(True)
        logging_tuple = agent.update_net(buffer_items)
        torch.set_grad_enabled(False)

        evaluator.evaluate_and_save(agent.act, args.horizon_len, logging_tuple)
        if (evaluator.total_step > args.break_step) or os.path.exists(f"{args.cwd}/stop"):
            torch.save(agent.act.state_dict(), args.cwd + '/actor.pth')
            break  # stop training when reach `break_step` or `mkdir cwd/stop`


def render_agent(env_class, env_args: dict, net_dims: [int], agent_class, actor_path: str, render_times: int = 8):
    env = build_env(env_class, env_args)

    state_dim = env_args['state_dim']
    action_dim = env_args['action_dim']
    agent = agent_class(net_dims, state_dim, action_dim, gpu_id=-1)
    actor = agent.act

    print(f"| render and load actor from: {actor_path}")
    actor.load_state_dict(torch.load(actor_path, map_location=lambda storage, loc: storage))
    for i in range(render_times):
        cumulative_reward, episode_step = get_rewards_and_steps(env, actor, if_render=True)
        print(f"|{i:4}  cumulative_reward {cumulative_reward:9.3f}  episode_step {episode_step:5.0f}")


class Evaluator:
    def __init__(self, eval_env, eval_per_step: int = 1e4, eval_times: int = 8, cwd: str = '.'):
        self.cwd = cwd
        self.env_eval = eval_env
        self.eval_step = 0
        self.total_step = 0
        self.start_time = time.time()
        self.eval_times = eval_times  # number of times that get episodic cumulative return
        self.eval_per_step = eval_per_step  # evaluate the agent per training steps

        self.recorder = []
        print(f"\n| `step`: Number of samples, or total training steps, or running times of `env.step()`."
              f"\n| `time`: Time spent from the start of training to this moment."
              f"\n| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode."
              f"\n| `avgS`: Average of steps in an episode."
              f"\n| `objC`: Objective of Critic network. Or call it loss function of critic network."
              f"\n| `objA`: Objective of Actor network. It is the average Q value of the critic network."
              f"\n| {'step':>8}  {'time':>8}  | {'avgR':>8}  {'stdR':>6}  {'avgS':>6}  | {'objC':>8}  {'objA':>8}")

    def evaluate_and_save(self, actor, horizon_len: int, logging_tuple: tuple):
        self.total_step += horizon_len
        if self.eval_step + self.eval_per_step > self.total_step:
            return
        self.eval_step = self.total_step

        rewards_steps_ary = [get_rewards_and_steps(self.env_eval, actor) for _ in range(self.eval_times)]
        rewards_steps_ary = np.array(rewards_steps_ary, dtype=np.float32)
        avg_r = rewards_steps_ary[:, 0].mean()  # average of cumulative rewards
        std_r = rewards_steps_ary[:, 0].std()  # std of cumulative rewards
        avg_s = rewards_steps_ary[:, 1].mean()  # average of steps in an episode

        used_time = time.time() - self.start_time
        self.recorder.append((self.total_step, used_time, avg_r))

        print(f"| {self.total_step:8.2e}  {used_time:8.0f}  "
              f"| {avg_r:8.2f}  {std_r:6.2f}  {avg_s:6.0f}  "
              f"| {logging_tuple[0]:8.2f}  {logging_tuple[1]:8.2f}")


def get_rewards_and_steps(env, actor, if_render: bool = False) -> (float, int):  # cumulative_rewards and episode_steps
    device = next(actor.parameters()).device  # net.parameters() is a Python generator.

    state, _ = env.reset()
    episode_steps = 0
    cumulative_returns = 0.0  # sum of rewards in an episode
    for episode_steps in range(12345):
        tensor_state = torch.as_tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
        tensor_action = actor(tensor_state)
        action = tensor_action.detach().cpu().numpy()[0]  # not need detach(), because using torch.no_grad() outside
        state, reward, done, _, _ = env.step(action)
        cumulative_returns += reward

        if if_render:
            env.render()
        if done:
            break
    return cumulative_returns, episode_steps + 1

## DRL Agent Class

In [ ]:
from __future__ import annotations

import torch
# from elegantrl.agents import AgentA2C

MODELS = {"ppo": AgentPPO}
OFF_POLICY_MODELS = ["ddpg", "td3", "sac"]
ON_POLICY_MODELS = ["ppo"]
# MODEL_KWARGS = {x: config.__dict__[f"{x.upper()}_PARAMS"] for x in MODELS.keys()}
#
# NOISE = {
#     "normal": NormalActionNoise,
#     "ornstein_uhlenbeck": OrnsteinUhlenbeckActionNoise,
# }


class DRLAgent:
    """Implementations of DRL algorithms
    Attributes
    ----------
        env: gym environment class
            user-defined class
    Methods
    -------
        get_model()
            setup DRL algorithms
        train_model()
            train DRL algorithms in a train dataset
            and output the trained model
        DRL_prediction()
            make a prediction in a test dataset and get results
    """

    def __init__(self, env, price_array, tech_array, turbulence_array, account_config=None):
        self.env = env
        self.price_array = price_array
        self.tech_array = tech_array
        self.turbulence_array = turbulence_array
        self.account_config = account_config or {}

    def get_model(self, model_name, model_kwargs):
        env_config = {
            "price_array": self.price_array,
            "tech_array": self.tech_array,
            "turbulence_array": self.turbulence_array,
            "if_train": True,
        }
        environment = self.env(config=env_config, **self.account_config)
        env_args = {'config': env_config,
              'env_name': environment.env_name,
              'state_dim': environment.state_dim,
              'action_dim': environment.action_dim,
              'if_discrete': False,
              **self.account_config}
        agent = MODELS[model_name]
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        model = Config(agent_class=agent, env_class=self.env, env_args=env_args)
        model.if_off_policy = model_name in OFF_POLICY_MODELS
        if model_kwargs is not None:
            try:
                model.learning_rate = model_kwargs["learning_rate"]
                model.batch_size = model_kwargs["batch_size"]
                model.gamma = model_kwargs["gamma"]
                model.seed = model_kwargs["seed"]
                model.net_dims = model_kwargs["net_dimension"]
                model.target_step = model_kwargs["target_step"]
                model.eval_gap = model_kwargs["eval_gap"]
                model.eval_times = model_kwargs["eval_times"]
            except BaseException:
                raise ValueError(
                    "Fail to read arguments, please check 'model_kwargs' input."
                )
        return model

    def train_model(self, model, cwd, total_timesteps=5000):
        model.cwd = cwd
        model.break_step = total_timesteps
        train_agent(model)

    @staticmethod
    def DRL_prediction(model_name, cwd, net_dimension, environment):
        if model_name not in MODELS:
            raise NotImplementedError("NotImplementedError")
        agent_class = MODELS[model_name]
        environment.env_num = 1
        # Force CPU usage with gpu_id=-1
        agent = agent_class(net_dimension, environment.state_dim, environment.action_dim, gpu_id=-1)
        actor = agent.act
        # load agent
        try:
            cwd = cwd + '/actor.pth'
            print(f"| load actor from: {cwd}")
            # Force CPU loading with map_location='cpu'
            actor.load_state_dict(torch.load(cwd, map_location='cpu'))
            act = actor
            device = agent.device
        except BaseException:
            raise ValueError("Fail to load agent!")

        # test on the testing env
        _torch = torch
        state, _ = environment.reset()
        episode_returns = []  # the cumulative_return / initial_account
        episode_total_assets = [environment.initial_total_asset]
        with _torch.no_grad():
            for i in range(environment.max_step):
                s_tensor = _torch.as_tensor((state,), device=device)
                a_tensor = act(s_tensor)  # action_tanh = act.forward()
                action = (
                    a_tensor.detach().cpu().numpy()[0]
                )  # not need detach(), because with torch.no_grad() outside
                state, reward, done, _, _ = environment.step(action)

                total_asset = (
                    environment.amount
                    + (
                        environment.price_ary[environment.day] * environment.stocks
                    ).sum()
                )
                episode_total_assets.append(total_asset)
                episode_return = total_asset / environment.initial_total_asset
                episode_returns.append(episode_return)
                if done:
                    break
        print("Test Finished!")
        # return episode total_assets on testing data
        print("episode_return", episode_return)
        return episode_total_assets

## Train & Test Functions

In [ ]:
from __future__ import annotations

from finrl.config import ERL_PARAMS
from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import SAC_PARAMS
from finrl.config import TRAIN_END_DATE
from finrl.config import TRAIN_START_DATE
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.data_processor import DataProcessor

# === TRANSACTION COST CONFIGURATION ===
TRANSACTION_COSTS = {
    "buy_cost_pct": 0.001,   # 0.1% for spread + slippage
    "sell_cost_pct": 0.001,
}


def train(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):
    # download data
    dp = DataProcessor(data_source, **kwargs)
    data = dp.download_data(ticker_list, start_date, end_date, time_interval)
    data = dp.clean_data(data)
    data = dp.add_technical_indicator(data, technical_indicator_list)
    if if_vix:
        data = dp.add_vix(data)
    else:
        data = dp.add_turbulence(data)
    price_array, tech_array, turbulence_array = dp.df_to_array(data, if_vix)
    
    # Include transaction costs
    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": True,
        **TRANSACTION_COSTS,
    }

    # Account configuration (initial_capital, max_stock, reward_scaling)
    account_config = kwargs.get("account_config", {})
    env_instance = env(config=env_config, **account_config)

    # read parameters
    cwd = kwargs.get("cwd", "./" + str(model_name))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        break_step = kwargs.get("break_step", 1e6)
        erl_params = kwargs.get("erl_params")
        agent = DRLAgent_erl(
            env=env,
            price_array=price_array,
            tech_array=tech_array,
            turbulence_array=turbulence_array,
            account_config=account_config,
        )
        model = agent.get_model(model_name, model_kwargs=erl_params)
        trained_model = agent.train_model(
            model=model, cwd=cwd, total_timesteps=break_step
        )

In [ ]:
from __future__ import annotations

from finrl.config import INDICATORS
from finrl.config import RLlib_PARAMS
from finrl.config import TEST_END_DATE
from finrl.config import TEST_START_DATE
from finrl.config_tickers import DOW_30_TICKER

def test(
    start_date,
    end_date,
    ticker_list,
    data_source,
    time_interval,
    technical_indicator_list,
    drl_lib,
    env,
    model_name,
    if_vix=True,
    **kwargs,
):

    from finrl.meta.data_processor import DataProcessor

    dp = DataProcessor(data_source, **kwargs)
    data = dp.download_data(ticker_list, start_date, end_date, time_interval)
    data = dp.clean_data(data)
    data = dp.add_technical_indicator(data, technical_indicator_list)

    if if_vix:
        data = dp.add_vix(data)
    else:
        data = dp.add_turbulence(data)
    price_array, tech_array, turbulence_array = dp.df_to_array(data, if_vix)

    # Include transaction costs
    # Account configuration (initial_capital, max_stock, reward_scaling)
    account_config = kwargs.get("account_config", {})
    env_config = {
        "price_array": price_array,
        "tech_array": tech_array,
        "turbulence_array": turbulence_array,
        "if_train": False,
        **TRANSACTION_COSTS,
    }
    env_instance = env(config=env_config, **account_config)

    net_dimension = kwargs.get("net_dimension", 2**7)
    cwd = kwargs.get("cwd", "./" + str(model_name))
    print("price_array: ", len(price_array))

    if drl_lib == "elegantrl":
        DRLAgent_erl = DRLAgent
        episode_total_assets = DRLAgent_erl.DRL_prediction(
            model_name=model_name,
            cwd=cwd,
            net_dimension=net_dimension,
            environment=env_instance,
        )
        return episode_total_assets


# === CORRECTED METRIC CALCULATIONS ===
def calculate_metrics(account_values, periods_per_day=390):
    """
    Calculate properly annualized Sharpe ratio and correct max drawdown.
    """
    account_values = np.array(account_values)
    
    total_return = (account_values[-1] / account_values[0]) - 1
    
    # Resample to daily for meaningful Sharpe
    daily_values = account_values[::periods_per_day]
    if len(daily_values) < 2:
        daily_values = account_values
    
    daily_returns = np.diff(daily_values) / daily_values[:-1]
    
    # Annualized Sharpe
    if len(daily_returns) > 1 and np.std(daily_returns) > 1e-10:
        sharpe_ratio = np.mean(daily_returns) / np.std(daily_returns) * np.sqrt(252)
    else:
        sharpe_ratio = 0.0
    
    # Correct max drawdown
    running_max = np.maximum.accumulate(account_values)
    drawdowns = (account_values - running_max) / running_max
    max_drawdown = np.min(drawdowns)
    
    # Calmar ratio
    trading_days = len(account_values) / periods_per_day
    annualized_return = (1 + total_return) ** (252 / max(trading_days, 1)) - 1
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    return {
        'total_return': total_return,
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar_ratio,
        'trading_days': trading_days
    }

## Import Dow Jones 30 Symbols

In [6]:
# Remove WBA since it has no data available from Alpaca
ticker_list = [ticker for ticker in DOW_30_TICKER if ticker != 'WBA']
action_dim = len(ticker_list)
print(f"Using {action_dim} stocks (removed WBA - no data available)")
print(f"Ticker list: {ticker_list}")

Using 29 stocks (removed WBA - no data available)
Ticker list: ['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WMT', 'DIS', 'DOW']


In [7]:
print(ticker_list)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WMT', 'DIS', 'DOW']


In [8]:
print(INDICATORS)

['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']


## Calculate the DRL state dimension manually for paper trading

In [9]:
# amount + (turbulence, turbulence_bool) + (price, shares, cd (holding time)) * stock_dim + tech_dim
# Updated to 29 stocks (removed WBA)
state_dim = 1 + 2 + 3 * action_dim + len(INDICATORS) * action_dim
print(f"State dimension: {state_dim} (for {action_dim} stocks)")

State dimension: 322 (for 29 stocks)


In [ ]:
state_dim

In [ ]:
action_dim

## Get the API Keys Ready

In [10]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.environ.get('ALPACA_API_KEY')
API_SECRET = os.environ.get('ALPACA_API_SECRET')
API_BASE_URL = os.environ.get('ALPACA_API_BASE_URL', 'https://paper-api.alpaca.markets')
data_url = os.environ.get('ALPACA_DATA_URL', 'wss://data.alpaca.markets')

env = StockTradingEnv

## Account Size Configuration

Configure `INITIAL_CAPITAL` and `MAX_STOCK` for your trading account tier.

**Key insight from ElegantRL author:** `reward_scaling` has minimal effect on PPO due to advantage normalization:
```python
buf_advantage = (buf_advantage - buf_advantage.mean()) / (buf_advantage.std() + 1e-5)
```
It mainly affects SAC. `episode_return = total_asset / initial_capital` is ratio-based and account-size invariant.

The critical parameters for different account sizes are:
- **`INITIAL_CAPITAL`**: Starting cash for training environment
- **`MAX_STOCK`**: Max shares per trade action (scales with account size)

In [ ]:
# === ACCOUNT SIZE CONFIGURATION ===
# Uncomment ONE preset below, or set custom values.
#
# reward_scaling has minimal effect on PPO (advantage normalization).
# episode_return = total_asset / initial_capital (ratio-based, account-size invariant).
# The key parameters are INITIAL_CAPITAL and MAX_STOCK.

# --- Preset: $1M (original default) ---
INITIAL_CAPITAL = 1_000_000
MAX_STOCK = 1e2

# --- Preset: $100K ---
#INITIAL_CAPITAL = 100_000
#MAX_STOCK = 1e2

# --- Preset: $50K ---
#INITIAL_CAPITAL = 50_000
#MAX_STOCK = 1e2

# --- Preset: $10K ---
#INITIAL_CAPITAL = 10_000
#MAX_STOCK = 10

REWARD_SCALING = 2**-11  # Default; minimal impact on PPO

ACCOUNT_CONFIG = {
    "initial_capital": INITIAL_CAPITAL,
    "max_stock": MAX_STOCK,
    "reward_scaling": REWARD_SCALING,
}

print("=" * 60)
print("ACCOUNT CONFIGURATION")
print("=" * 60)
print(f"Initial Capital:  ${INITIAL_CAPITAL:>12,.0f}")
print(f"Max Stock/Trade:  {MAX_STOCK:>12.0f}")
print(f"Reward Scaling:   {REWARD_SCALING:>12.6f} (minimal effect on PPO)")
print("=" * 60)

# Improved Hyperparameter Optimization with Walk-Forward Validation

Key improvements over original approach:
1. **Walk-Forward Validation**: Test on multiple time windows to ensure robustness
2. **Corrected Sharpe Ratio**: Properly annualized using daily returns
3. **Corrected Max Drawdown**: Fixed calculation error
4. **Transaction Costs**: Included in training environment
5. **Smaller Network Options**: To reduce overfitting risk
6. **Longer Training Period**: 4+ months per window

## Walk-Forward Validation Windows

Instead of a single train/test split, we use multiple rolling windows to:
- Reduce selection bias
- Test across different market regimes
- Get statistically meaningful results

In [11]:
import itertools
import json
from datetime import datetime as dt

# === WALK-FORWARD VALIDATION WINDOWS ===
# Each window: (train_start, train_end, val_start, val_end)
# Using ~4 months training, ~1 month validation per window
VALIDATION_WINDOWS = [
    ('2025-02-01', '2025-05-31', '2025-06-01', '2025-06-30'),  # Window 1
    ('2025-04-01', '2025-07-31', '2025-08-01', '2025-08-31'),  # Window 2  
    ('2025-06-01', '2025-09-30', '2025-10-01', '2025-10-31'),  # Window 3
]

print(f"Using {len(VALIDATION_WINDOWS)} walk-forward validation windows:")
for i, (ts, te, vs, ve) in enumerate(VALIDATION_WINDOWS):
    print(f"  Window {i+1}: Train {ts} to {te} → Validate {vs} to {ve}")

# === IMPROVED PARAMETER GRID ===
# Smaller networks to reduce overfitting
# Lower gamma values for shorter trading horizons
param_grid = {
    'learning_rate': [1e-5, 3e-5, 1e-4],
    'gamma': [0.90, 0.95, 0.97],  # Lower gamma for intraday trading
    'net_dimension': [[32, 16], [64, 32], [128, 64]],  # Smaller networks
    'batch_size': [256, 512, 1024]  # Smaller batches for more updates
}

# Generate all combinations
param_combinations = list(itertools.product(
    param_grid['learning_rate'],
    param_grid['gamma'],
    param_grid['net_dimension'],
    param_grid['batch_size']
))

print(f"\nTotal hyperparameter combinations: {len(param_combinations)}")
print(f"Total experiments (combinations × windows): {len(param_combinations) * len(VALIDATION_WINDOWS)}")

Using 3 walk-forward validation windows:
  Window 1: Train 2025-02-01 to 2025-05-31 → Validate 2025-06-01 to 2025-06-30
  Window 2: Train 2025-04-01 to 2025-07-31 → Validate 2025-08-01 to 2025-08-31
  Window 3: Train 2025-06-01 to 2025-09-30 → Validate 2025-10-01 to 2025-10-31

Total hyperparameter combinations: 81
Total experiments (combinations × windows): 243


## Run Grid Search (WARNING: Time-consuming!)

In [15]:
# === CHECKPOINT REPAIR UTILITY ===
# Run this cell ONLY if you need to fix the checkpoint after a connection failure
# This removes failed experiments from the checkpoint so they will be retried

import os
import pickle
import pandas as pd

CHECKPOINT_DIR = './hpo_checkpoints'
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'hpo_checkpoint.pkl')

def repair_checkpoint():
    """Remove failed experiments from checkpoint, keeping only successful ones."""
    if not os.path.exists(CHECKPOINT_FILE):
        print("❌ No checkpoint file found!")
        return
    
    with open(CHECKPOINT_FILE, 'rb') as f:
        checkpoint = pickle.load(f)
    
    results = checkpoint['results']
    completed = checkpoint['completed_experiments']
    
    # Count successes and failures
    successful = [r for r in results if 'error' not in r]
    failed = [r for r in results if 'error' in r]
    
    print(f"Current checkpoint status:")
    print(f"  Total experiments marked complete: {len(completed)}")
    print(f"  Successful experiments: {len(successful)}")
    print(f"  Failed experiments: {len(failed)}")
    
    if len(failed) == 0:
        print("\n✅ No failed experiments to remove!")
        return
    
    # Get exp_ids for successful experiments only
    successful_ids = set()
    for r in successful:
        exp_key = r['exp_key']
        window = r['window']
        exp_id = f"{exp_key}_window{window}"
        successful_ids.add(exp_id)
    
    # Rebuild aggregated_results from successful only
    from collections import defaultdict
    new_aggregated = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    
    for r in successful:
        exp_key = r['exp_key']
        new_aggregated[exp_key]['sharpe_ratios'].append(r['sharpe_ratio'])
        new_aggregated[exp_key]['returns'].append(r['total_return'])
        new_aggregated[exp_key]['max_drawdowns'].append(r['max_drawdown'])
        new_aggregated[exp_key]['successful_windows'] += 1
        new_aggregated[exp_key]['params'] = {
            'learning_rate': r['learning_rate'],
            'gamma': r['gamma'],
            'net_dimension': r['net_dimension'],
            'batch_size': r['batch_size']
        }
    
    # Create repaired checkpoint
    repaired = {
        'results': successful,
        'aggregated_results': dict(new_aggregated),
        'completed_experiments': list(successful_ids),
        'timestamp': checkpoint['timestamp'] + ' (REPAIRED)'
    }
    
    # Backup old checkpoint
    backup_file = CHECKPOINT_FILE + '.backup'
    os.rename(CHECKPOINT_FILE, backup_file)
    print(f"\n📦 Old checkpoint backed up to: {backup_file}")
    
    # Save repaired checkpoint
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(repaired, f)
    
    print(f"✅ Checkpoint repaired!")
    print(f"   Kept {len(successful)} successful experiments")
    print(f"   Removed {len(failed)} failed experiments")
    print(f"\n🔄 Now re-run the grid search cell to resume from experiment {len(successful)+1}")

# Run the repair
repair_checkpoint()

Current checkpoint status:
  Total experiments marked complete: 243
  Successful experiments: 41
  Failed experiments: 202

📦 Old checkpoint backed up to: ./hpo_checkpoints/hpo_checkpoint.pkl.backup
✅ Checkpoint repaired!
   Kept 41 successful experiments
   Removed 202 failed experiments

🔄 Now re-run the grid search cell to resume from experiment 42


In [ ]:
# Walk-Forward Grid Search with CHECKPOINTING + CONNECTION MONITORING
# This tests each hyperparameter combination across ALL validation windows
# and aggregates results to find the most robust configuration
# 
# CHECKPOINT FEATURE: Saves progress after each experiment. If the kernel crashes
# or disconnects, simply re-run this cell to resume from where you left off.
#
# CONNECTION MONITORING: Detects connection failures and exits gracefully instead
# of continuing to fail. Re-run the cell after connection is restored.

import os
import time
import json
import pickle
import socket
import requests
from collections import defaultdict

# === CHECKPOINT CONFIGURATION ===
CHECKPOINT_DIR = './hpo_checkpoints'
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'hpo_checkpoint.pkl')
RESULTS_BACKUP_FILE = os.path.join(CHECKPOINT_DIR, 'results_backup.csv')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# === CONNECTION MONITORING ===
CONSECUTIVE_FAILURE_LIMIT = 2  # Exit after this many consecutive failures
CONNECTION_CHECK_TIMEOUT = 10  # seconds

def check_internet_connection():
    """Check if we have internet connectivity."""
    try:
        # Try to connect to Alpaca API
        response = requests.get('https://api.alpaca.markets/v2/clock', 
                               headers={'APCA-API-KEY-ID': API_KEY, 
                                       'APCA-API-SECRET-KEY': API_SECRET},
                               timeout=CONNECTION_CHECK_TIMEOUT)
        return response.status_code == 200
    except:
        pass
    
    # Fallback: try common endpoints
    for host in ['8.8.8.8', '1.1.1.1']:
        try:
            socket.create_connection((host, 53), timeout=3)
            return True
        except:
            continue
    return False

def is_connection_error(error_msg):
    """Check if an error is likely due to connection issues."""
    connection_keywords = [
        'connection', 'timeout', 'timed out', 'network', 'unreachable',
        'refused', 'reset', 'socket', 'ssl', 'certificate', 'dns',
        'name resolution', 'temporarily unavailable', 'no route',
        'connection reset', 'broken pipe', 'eof', 'connection aborted'
    ]
    error_lower = str(error_msg).lower()
    return any(keyword in error_lower for keyword in connection_keywords)

def save_checkpoint(results, aggregated_results, completed_experiments):
    """Save current progress to checkpoint file."""
    checkpoint = {
        'results': results,
        'aggregated_results': dict(aggregated_results),
        'completed_experiments': completed_experiments,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(checkpoint, f)
    
    # Also save results as CSV backup
    if results:
        results_df = pd.DataFrame([r for r in results if 'error' not in r])
        if len(results_df) > 0:
            results_df.to_csv(RESULTS_BACKUP_FILE, index=False)
    
    print(f"   💾 Checkpoint saved ({len(completed_experiments)} experiments complete)")

def load_checkpoint():
    """Load checkpoint if it exists."""
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'rb') as f:
                checkpoint = pickle.load(f)
            print(f"📂 Checkpoint found from {checkpoint['timestamp']}")
            print(f"   Resuming from {len(checkpoint['completed_experiments'])} completed experiments")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Failed to load checkpoint: {e}")
            return None
    return None

# Pre-flight connection check
print("🔌 Checking connection to Alpaca API...")
if not check_internet_connection():
    print("❌ Cannot connect to Alpaca API!")
    print("   Please check your internet connection and try again.")
    raise ConnectionError("No connection to Alpaca API. Please restore connection and re-run.")
print("✅ Connection OK\n")

# Try to load existing checkpoint
checkpoint = load_checkpoint()

if checkpoint:
    results = checkpoint['results']
    # Reconstruct defaultdict from regular dict
    aggregated_results = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    for key, value in checkpoint['aggregated_results'].items():
        aggregated_results[key] = value
    completed_experiments = set(checkpoint['completed_experiments'])
    print(f"✅ Resuming HPO - {len(completed_experiments)} experiments already complete\n")
else:
    results = []
    aggregated_results = defaultdict(lambda: {
        'sharpe_ratios': [],
        'returns': [],
        'max_drawdowns': [],
        'successful_windows': 0
    })
    completed_experiments = set()
    print("🆕 Starting fresh HPO run\n")

# Option 1: Test a subset (recommended for quick testing)
# test_combinations = param_combinations[:3]

# Option 2: Test all combinations
test_combinations = param_combinations

# Option 3: Random search - test N random combinations
# import random
# random.seed(312)
# N_RANDOM_TESTS = 27
# test_combinations = random.sample(param_combinations, min(N_RANDOM_TESTS, len(param_combinations)))

total_experiments = len(test_combinations) * len(VALIDATION_WINDOWS)
remaining = total_experiments - len(completed_experiments)

print(f"Testing {len(test_combinations)} combinations across {len(VALIDATION_WINDOWS)} windows")
print(f"Total experiments: {total_experiments}")
print(f"Already completed: {len(completed_experiments)}")
print(f"Remaining: {remaining}")

current_exp = 0
consecutive_failures = 0  # Track consecutive connection failures
graceful_exit = False

for lr, gamma, net_dim, bs in test_combinations:
    if graceful_exit:
        break
        
    exp_key = f"lr{lr:.0e}_g{gamma}_net{'x'.join(map(str, net_dim))}_bs{bs}"
    
    for window_idx, (train_start, train_end, val_start, val_end) in enumerate(VALIDATION_WINDOWS):
        if graceful_exit:
            break
            
        current_exp += 1
        
        # Create unique experiment ID
        exp_id = f"{exp_key}_window{window_idx+1}"
        
        # Skip if already completed
        if exp_id in completed_experiments:
            print(f"⏭️  Skipping {current_exp}/{total_experiments} (already complete): {exp_id}")
            continue
        
        print(f"\n{'='*70}")
        print(f"Experiment {current_exp}/{total_experiments} ({len(completed_experiments)+1} running)")
        print(f"Config: LR={lr:.0e}, Gamma={gamma}, Net={net_dim}, Batch={bs}")
        print(f"Window {window_idx+1}: Train {train_start}→{train_end} | Val {val_start}→{val_end}")
        print(f"{'='*70}")
        
        exp_params = {
            "learning_rate": lr,
            "batch_size": bs,
            "gamma": gamma,
            "seed": 312,
            "net_dimension": net_dim,
            "target_step": 5000,
            "eval_gap": 30,
            "eval_times": 1
        }
        
        exp_cwd = f'./hpo_experiments/{exp_key}/window_{window_idx+1}'
        os.makedirs(exp_cwd, exist_ok=True)
        
        try:
            # Train on training window
            train(
                start_date=train_start,
                end_date=train_end,
                ticker_list=ticker_list,
                data_source='alpaca',
                time_interval='1Min',
                technical_indicator_list=INDICATORS,
                drl_lib='elegantrl',
                env=env,
                model_name='ppo',
                if_vix=True,
                API_KEY=API_KEY,
                API_SECRET=API_SECRET,
                API_BASE_URL=API_BASE_URL,
                erl_params=exp_params,
                cwd=exp_cwd,
                break_step=1e5,
                account_config=ACCOUNT_CONFIG
            )
            
            # Test on validation window
            account_values = test(
                start_date=val_start,
                end_date=val_end,
                ticker_list=ticker_list,
                data_source='alpaca',
                time_interval='1Min',
                technical_indicator_list=INDICATORS,
                drl_lib='elegantrl',
                env=env,
                model_name='ppo',
                if_vix=True,
                API_KEY=API_KEY,
                API_SECRET=API_SECRET,
                API_BASE_URL=API_BASE_URL,
                cwd=exp_cwd,
                net_dimension=net_dim,
                account_config=ACCOUNT_CONFIG
            )
            
            # Use corrected metrics calculation
            metrics = calculate_metrics(account_values)
            
            result = {
                'learning_rate': lr,
                'gamma': gamma,
                'net_dimension': net_dim,
                'batch_size': bs,
                'window': window_idx + 1,
                'train_start': train_start,
                'train_end': train_end,
                'val_start': val_start,
                'val_end': val_end,
                'total_return': metrics['total_return'],
                'sharpe_ratio': metrics['sharpe_ratio'],
                'max_drawdown': metrics['max_drawdown'],
                'final_value': account_values[-1],
                'exp_key': exp_key
            }
            results.append(result)
            
            # Aggregate for this config
            aggregated_results[exp_key]['sharpe_ratios'].append(metrics['sharpe_ratio'])
            aggregated_results[exp_key]['returns'].append(metrics['total_return'])
            aggregated_results[exp_key]['max_drawdowns'].append(metrics['max_drawdown'])
            aggregated_results[exp_key]['successful_windows'] += 1
            aggregated_results[exp_key]['params'] = {
                'learning_rate': lr, 'gamma': gamma, 
                'net_dimension': net_dim, 'batch_size': bs
            }
            
            print(f"\n✅ Window {window_idx+1} Results:")
            print(f"   Total Return: {metrics['total_return']:.2%}")
            print(f"   Sharpe Ratio: {metrics['sharpe_ratio']:.3f}")
            print(f"   Max Drawdown: {metrics['max_drawdown']:.2%}")
            
            # Mark as completed and save checkpoint
            completed_experiments.add(exp_id)
            save_checkpoint(results, aggregated_results, list(completed_experiments))
            
            # Reset consecutive failures on success
            consecutive_failures = 0
            
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Experiment failed: {error_msg}")
            
            # Check if this is a connection error
            if is_connection_error(error_msg):
                consecutive_failures += 1
                print(f"⚠️  Connection error detected ({consecutive_failures}/{CONSECUTIVE_FAILURE_LIMIT})")
                
                # Verify connection is actually down
                if not check_internet_connection():
                    print(f"\n🔌 CONNECTION LOST!")
                    print(f"   Exiting gracefully to preserve progress.")
                    print(f"   Successfully completed: {len(completed_experiments)} experiments")
                    print(f"   Remaining: {total_experiments - len(completed_experiments)} experiments")
                    print(f"\n   ➡️  Restore connection and re-run this cell to resume.")
                    graceful_exit = True
                    break
                
                if consecutive_failures >= CONSECUTIVE_FAILURE_LIMIT:
                    print(f"\n⚠️  {CONSECUTIVE_FAILURE_LIMIT} consecutive failures detected!")
                    print(f"   Pausing to check connection...")
                    
                    if not check_internet_connection():
                        print(f"\n🔌 CONNECTION LOST!")
                        print(f"   Exiting gracefully to preserve progress.")
                        graceful_exit = True
                        break
                    else:
                        print(f"   Connection seems OK, continuing...")
                        consecutive_failures = 0
            else:
                # Non-connection error - don't count towards limit, but don't mark as complete
                # This allows retry of legitimately failed experiments
                consecutive_failures = 0
            
            # Do NOT mark connection failures as complete - they should be retried
            if not is_connection_error(error_msg):
                results.append({
                    'learning_rate': lr, 'gamma': gamma,
                    'net_dimension': net_dim, 'batch_size': bs,
                    'window': window_idx + 1, 'error': error_msg,
                    'exp_key': exp_key
                })
                completed_experiments.add(exp_id)
                save_checkpoint(results, aggregated_results, list(completed_experiments))

if graceful_exit:
    print(f"\n{'='*70}")
    print(f"⏸️  HPO PAUSED DUE TO CONNECTION LOSS")
    print(f"{'='*70}")
    print(f"Progress saved: {len(completed_experiments)}/{total_experiments} experiments")
    print(f"\nTo resume: Restore internet connection and re-run this cell")
else:
    print(f"\n{'='*70}")
    print(f"Walk-forward grid search complete!")
    print(f"Tested {len(test_combinations)} configs across {len(VALIDATION_WINDOWS)} windows")
    print(f"Successfully completed: {len([r for r in results if 'error' not in r])}")
    print(f"Failed: {len([r for r in results if 'error' in r])}")
    print(f"{'='*70}")

# Final save
save_checkpoint(results, aggregated_results, list(completed_experiments))

🔌 Checking connection to Alpaca API...
✅ Connection OK

📂 Checkpoint found from 2026-01-30 17:11:49
   Resuming from 212 completed experiments
✅ Resuming HPO - 212 experiments already complete

Testing 81 combinations across 3 windows
Total experiments: 243
Already completed: 212
Remaining: 31
⏭️  Skipping 1/243 (already complete): lr1e-05_g0.9_net32x16_bs256_window1
⏭️  Skipping 2/243 (already complete): lr1e-05_g0.9_net32x16_bs256_window2
⏭️  Skipping 3/243 (already complete): lr1e-05_g0.9_net32x16_bs256_window3
⏭️  Skipping 4/243 (already complete): lr1e-05_g0.9_net32x16_bs512_window1
⏭️  Skipping 5/243 (already complete): lr1e-05_g0.9_net32x16_bs512_window2
⏭️  Skipping 6/243 (already complete): lr1e-05_g0.9_net32x16_bs512_window3
⏭️  Skipping 7/243 (already complete): lr1e-05_g0.9_net32x16_bs1024_window1
⏭️  Skipping 8/243 (already complete): lr1e-05_g0.9_net32x16_bs1024_window2
⏭️  Skipping 9/243 (already complete): lr1e-05_g0.9_net32x16_bs1024_window3
⏭️  Skipping 10/243 (alread

## Analyze Results

In [19]:
# Analyze Walk-Forward Results
# The key is finding configurations that perform well CONSISTENTLY across windows

# Create aggregated summary
agg_summary = []
for exp_key, data in aggregated_results.items():
    if data['successful_windows'] == len(VALIDATION_WINDOWS):  # Only use configs that worked on all windows
        params = data['params']
        agg_summary.append({
            'exp_key': exp_key,
            'learning_rate': params['learning_rate'],
            'gamma': params['gamma'],
            'net_dimension': str(params['net_dimension']),
            'batch_size': params['batch_size'],
            'mean_sharpe': np.mean(data['sharpe_ratios']),
            'std_sharpe': np.std(data['sharpe_ratios']),
            'min_sharpe': np.min(data['sharpe_ratios']),  # Worst case
            'mean_return': np.mean(data['returns']),
            'worst_drawdown': np.min(data['max_drawdowns']),  # Worst drawdown across windows
            'consistency': np.mean(data['sharpe_ratios']) / (np.std(data['sharpe_ratios']) + 0.1)  # Higher = more consistent
        })

agg_df = pd.DataFrame(agg_summary)

if len(agg_df) > 0:
    print("=" * 80)
    print("WALK-FORWARD AGGREGATED RESULTS")
    print(f"Showing configs that succeeded on all {len(VALIDATION_WINDOWS)} windows")
    print("=" * 80)
    
    # Sort by mean Sharpe
    print("\n📊 TOP 5 BY MEAN SHARPE RATIO (across all windows):")
    print("-" * 80)
    top_sharpe = agg_df.sort_values('mean_sharpe', ascending=False).head()
    print(top_sharpe[['learning_rate', 'gamma', 'net_dimension', 'batch_size', 
                      'mean_sharpe', 'std_sharpe', 'mean_return', 'worst_drawdown']].to_string())
    
    # Sort by consistency (robust performance)
    print("\n📊 TOP 5 BY CONSISTENCY (mean_sharpe / std_sharpe):")
    print("-" * 80)
    top_consistent = agg_df.sort_values('consistency', ascending=False).head()
    print(top_consistent[['learning_rate', 'gamma', 'net_dimension', 'batch_size',
                          'mean_sharpe', 'std_sharpe', 'consistency']].to_string())
    
    # Sort by worst-case Sharpe (most conservative choice)
    print("\n📊 TOP 5 BY MINIMUM SHARPE (best worst-case performance):")
    print("-" * 80)
    top_minsharp = agg_df.sort_values('min_sharpe', ascending=False).head()
    print(top_minsharp[['learning_rate', 'gamma', 'net_dimension', 'batch_size',
                        'min_sharpe', 'mean_sharpe', 'worst_drawdown']].to_string())
    
    # RECOMMENDED: Balance of performance and consistency
    print("\n" + "=" * 80)
    print("🏆 RECOMMENDED CONFIGURATION (by consistency):")
    print("=" * 80)
    best = agg_df.sort_values('consistency', ascending=False).iloc[0]
    print(f"Learning Rate: {best['learning_rate']:.0e}")
    print(f"Gamma: {best['gamma']}")
    print(f"Network: {best['net_dimension']}")
    print(f"Batch Size: {int(best['batch_size'])}")
    print(f"\nAggregated Performance (across {len(VALIDATION_WINDOWS)} validation windows):")
    print(f"  Mean Sharpe Ratio: {best['mean_sharpe']:.3f} ± {best['std_sharpe']:.3f}")
    print(f"  Mean Return: {best['mean_return']:.2%}")
    print(f"  Worst Max Drawdown: {best['worst_drawdown']:.2%}")
    print(f"  Consistency Score: {best['consistency']:.2f}")
    
    # Save all results
    results_df = pd.DataFrame([r for r in results if 'error' not in r])
    results_df.to_csv('./hpo_per_window_results.csv', index=False)
    agg_df.to_csv('./hpo_aggregated_results.csv', index=False)
    print("\n✅ Results saved to hpo_per_window_results.csv and hpo_aggregated_results.csv")
    
    # Create recommended ERL_PARAMS
    print("\n" + "=" * 80)
    print("📋 COPY THESE PARAMS TO v2 NOTEBOOK:")
    print("=" * 80)
    print(f'''
ERL_PARAMS = {{
    "learning_rate": {best['learning_rate']:.0e},
    "batch_size": {int(best['batch_size'])},
    "gamma": {best['gamma']},
    "seed": 312,
    "net_dimension": {best['net_dimension']},
    "target_step": 5000,
    "eval_gap": 30,
    "eval_times": 1
}}
''')
else:
    print("⚠️ No configurations succeeded on all validation windows.")
    print("Consider increasing N_RANDOM_TESTS or checking for data issues.")

WALK-FORWARD AGGREGATED RESULTS
Showing configs that succeeded on all 3 windows

📊 TOP 5 BY MEAN SHARPE RATIO (across all windows):
--------------------------------------------------------------------------------
    learning_rate  gamma net_dimension  batch_size  mean_sharpe  std_sharpe  mean_return  worst_drawdown
1         0.00001   0.90      [32, 16]         512     7.128213    3.970628     0.055338       -0.032055
65        0.00010   0.95      [32, 16]        1024     6.849263    4.864145     0.062994       -0.035568
64        0.00010   0.95      [32, 16]         512     6.594799    2.782708     0.054013       -0.041320
2         0.00001   0.90      [32, 16]        1024     6.560875    3.387610     0.067545       -0.043319
5         0.00001   0.90      [64, 32]        1024     6.318745    0.976285     0.062443       -0.038331

📊 TOP 5 BY CONSISTENCY (mean_sharpe / std_sharpe):
--------------------------------------------------------------------------------
    learning_rate  gamma

## Visualize Parameter Impact

In [20]:
import matplotlib.pyplot as plt

if len(agg_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Learning rate vs Mean Sharpe
    lr_groups = agg_df.groupby('learning_rate')['mean_sharpe'].mean()
    axes[0, 0].bar(range(len(lr_groups)), lr_groups.values, color='steelblue')
    axes[0, 0].set_xticks(range(len(lr_groups)))
    axes[0, 0].set_xticklabels([f'{lr:.0e}' for lr in lr_groups.index])
    axes[0, 0].set_title('Learning Rate vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Learning Rate')
    axes[0, 0].set_ylabel('Mean Sharpe Ratio')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Gamma vs Mean Sharpe
    gamma_groups = agg_df.groupby('gamma')['mean_sharpe'].mean()
    axes[0, 1].bar([str(g) for g in gamma_groups.index], gamma_groups.values, color='darkorange')
    axes[0, 1].set_title('Gamma vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Gamma')
    axes[0, 1].set_ylabel('Mean Sharpe Ratio')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Network size vs Mean Sharpe
    net_groups = agg_df.groupby('net_dimension')['mean_sharpe'].mean()
    axes[0, 2].bar(range(len(net_groups)), net_groups.values, color='forestgreen')
    axes[0, 2].set_xticks(range(len(net_groups)))
    axes[0, 2].set_xticklabels(net_groups.index, rotation=45)
    axes[0, 2].set_title('Network Size vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Network Dimensions')
    axes[0, 2].set_ylabel('Mean Sharpe Ratio')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Batch size vs Mean Sharpe
    bs_groups = agg_df.groupby('batch_size')['mean_sharpe'].mean()
    axes[1, 0].bar([str(int(b)) for b in bs_groups.index], bs_groups.values, color='purple')
    axes[1, 0].set_title('Batch Size vs Mean Sharpe', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Batch Size')
    axes[1, 0].set_ylabel('Mean Sharpe Ratio')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Mean Sharpe vs Consistency scatter
    axes[1, 1].scatter(agg_df['mean_sharpe'], agg_df['consistency'], 
                       c=agg_df['worst_drawdown'], cmap='RdYlGn', s=100, alpha=0.7)
    axes[1, 1].set_title('Mean Sharpe vs Consistency', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Mean Sharpe Ratio')
    axes[1, 1].set_ylabel('Consistency Score')
    axes[1, 1].grid(True, alpha=0.3)
    cbar = plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1])
    cbar.set_label('Worst Drawdown')
    
    # 6. Per-window performance for top 5 configs
    results_df = pd.DataFrame([r for r in results if 'error' not in r])
    top5_keys = agg_df.sort_values('consistency', ascending=False).head(5)['exp_key'].values
    for key in top5_keys:
        config_results = results_df[results_df['exp_key'] == key]
        axes[1, 2].plot(config_results['window'], config_results['sharpe_ratio'], 
                       marker='o', label=key[:25], linewidth=2)
    axes[1, 2].set_title('Top 5 Configs: Sharpe by Window', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Validation Window')
    axes[1, 2].set_ylabel('Sharpe Ratio')
    axes[1, 2].legend(fontsize=8, loc='best')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].set_xticks([1, 2, 3])
    
    plt.tight_layout()
    plt.savefig('./hpo_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Analysis plot saved to hpo_analysis.png")
else:
    print("No data to visualize.")

✅ Analysis plot saved to hpo_analysis.png


/tmp/ipykernel_354197/1846966496.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Final Backtest Validation

After finding the best hyperparameters via walk-forward validation, we perform a final backtest on a **completely held-out period** (November-December 2025) that was never seen during HPO.

This gives an unbiased estimate of out-of-sample performance before deploying to the v2 trading notebook.

In [21]:
# === FINAL BACKTEST CONFIGURATION ===
# Use the best hyperparameters from walk-forward validation
# Train on the combined HPO period, test on held-out period

# Get best config from results
if len(agg_df) > 0:
    best_config = agg_df.sort_values('consistency', ascending=False).iloc[0]
    
    BEST_PARAMS = {
        "learning_rate": float(best_config['learning_rate']),
        "batch_size": int(best_config['batch_size']),
        "gamma": float(best_config['gamma']),
        "seed": 312,
        "net_dimension": eval(best_config['net_dimension']),  # Convert string back to list
        "target_step": 5000,
        "eval_gap": 30,
        "eval_times": 1
    }
    
    print("=" * 70)
    print("FINAL BACKTEST CONFIGURATION")
    print("=" * 70)
    print(f"Best config from HPO: {best_config['exp_key']}")
    print(f"Parameters: {BEST_PARAMS}")
    print()
    
    # Training period: Feb-Oct 2025 (covers all HPO windows)
    BACKTEST_TRAIN_START = '2025-02-01'
    BACKTEST_TRAIN_END = '2025-10-31'
    
    # Test period: Nov-Dec 2025 (completely held-out from HPO)
    BACKTEST_TEST_START = '2025-11-01'
    BACKTEST_TEST_END = '2025-12-31'
    
    print(f"Training Period: {BACKTEST_TRAIN_START} to {BACKTEST_TRAIN_END}")
    print(f"Test Period: {BACKTEST_TEST_START} to {BACKTEST_TEST_END} (held-out)")
    print("=" * 70)
else:
    print("⚠️ No HPO results available. Run the grid search first.")

FINAL BACKTEST CONFIGURATION
Best config from HPO: lr3e-05_g0.9_net32x16_bs1024
Parameters: {'learning_rate': 3e-05, 'batch_size': 1024, 'gamma': 0.9, 'seed': 312, 'net_dimension': [32, 16], 'target_step': 5000, 'eval_gap': 30, 'eval_times': 1}

Training Period: 2025-02-01 to 2025-10-31
Test Period: 2025-11-01 to 2025-12-31 (held-out)


In [ ]:
# === TRAIN MODEL FOR BACKTEST ===
if len(agg_df) > 0:
    backtest_cwd = './backtest_final'
    
    print("Training model with best hyperparameters...")
    print(f"Period: {BACKTEST_TRAIN_START} to {BACKTEST_TRAIN_END}")
    
    train(
        start_date=BACKTEST_TRAIN_START,
        end_date=BACKTEST_TRAIN_END,
        ticker_list=ticker_list,
        data_source='alpaca',
        time_interval='1Min',
        technical_indicator_list=INDICATORS,
        drl_lib='elegantrl',
        env=env,
        model_name='ppo',
        if_vix=True,
        API_KEY=API_KEY,
        API_SECRET=API_SECRET,
        API_BASE_URL=API_BASE_URL,
        erl_params=BEST_PARAMS,
        cwd=backtest_cwd,
        break_step=2e5,  # More steps for full training
        account_config=ACCOUNT_CONFIG
    )
    
    print("\n✅ Training complete!")

Training model with best hyperparameters...
Period: 2025-02-01 to 2025-10-31
Alpaca successfully connected
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
The price of the first row for ticker AAPL is NaN. It will be filled with the first valid price.
The price of the first row for ticker AMGN is NaN. It will be filled with the first valid price.
The price of the first row for ticker AXP is NaN. It will be filled with the first valid price.
The price of the first row for ticker BA is NaN. It will be filled with the first valid price.
The price of the first row for ticker CAT is NaN. It will be filled with the first valid price.
The price of the first row for ticker CRM is NaN. It will be filled with the first valid price.
The price of the first row for ticker CSCO is NaN. It will be filled with the first valid price.
The price of the first row for ticker CVX is NaN. It will be filled with the first valid price.
The price of the firs

In [ ]:
# === RUN BACKTEST ON HELD-OUT PERIOD ===
if len(agg_df) > 0:
    print("Running backtest on held-out period...")
    print(f"Period: {BACKTEST_TEST_START} to {BACKTEST_TEST_END}")
    
    backtest_account_values = test(
        start_date=BACKTEST_TEST_START,
        end_date=BACKTEST_TEST_END,
        ticker_list=ticker_list,
        data_source='alpaca',
        time_interval='1Min',
        technical_indicator_list=INDICATORS,
        drl_lib='elegantrl',
        env=env,
        model_name='ppo',
        if_vix=True,
        API_KEY=API_KEY,
        API_SECRET=API_SECRET,
        API_BASE_URL=API_BASE_URL,
        cwd=backtest_cwd,
        net_dimension=BEST_PARAMS['net_dimension'],
        account_config=ACCOUNT_CONFIG
    )
    
    # Calculate metrics
    backtest_metrics = calculate_metrics(backtest_account_values)
    
    print("\n" + "=" * 70)
    print("📊 FINAL BACKTEST RESULTS (Held-Out Period)")
    print("=" * 70)
    print(f"Period: {BACKTEST_TEST_START} to {BACKTEST_TEST_END}")
    print(f"Trading Days: {backtest_metrics['trading_days']:.1f}")
    print()
    print(f"  Total Return:    {backtest_metrics['total_return']:>10.2%}")
    print(f"  Sharpe Ratio:    {backtest_metrics['sharpe_ratio']:>10.3f} (annualized)")
    print(f"  Max Drawdown:    {backtest_metrics['max_drawdown']:>10.2%}")
    print(f"  Calmar Ratio:    {backtest_metrics['calmar_ratio']:>10.3f}")
    print()
    print(f"  Initial Value:   ${backtest_account_values[0]:>12,.2f}")
    print(f"  Final Value:     ${backtest_account_values[-1]:>12,.2f}")
    print("=" * 70)

Running backtest on held-out period...
Period: 2025-11-01 to 2025-12-31
Alpaca successfully connected
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
ticker list complete
Start concat and rename
Data clean finished!
Started adding Indicators
Running Loop
Restore Timestamps
Finished adding Indicators
Data cleaning started
align start and end dates
produce full timestamp index
Start processing tickers
ticker list complete
Start concat and rename
Data clean finished!
price_array:  15990
| load actor from: ./backtest_final/actor.pth
Test Finished!
episode_return 1.0382999000377866

📊 FINAL BACKTEST RESULTS (Held-Out Period)
Period: 2025-11-01 to 2025-12-31
Trading Days: 41.0

  Total Return:         3.83%
  Sharpe Ratio:         1.997 (annualized)
  Max Drawdown:        -4.98%
  Calmar Ratio:         5.222

  Initial Value:   $1,000,000.00
  Final Value:     $1,038,299.90


In [24]:
# === VISUALIZE BACKTEST RESULTS ===
if len(agg_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Portfolio Value Over Time
    axes[0].plot(backtest_account_values, linewidth=2, color='blue')
    axes[0].axhline(y=backtest_account_values[0], color='gray', linestyle='--', alpha=0.7, label='Initial Value')
    axes[0].fill_between(range(len(backtest_account_values)), 
                         backtest_account_values[0], 
                         backtest_account_values, 
                         where=[v > backtest_account_values[0] for v in backtest_account_values],
                         alpha=0.3, color='green', label='Profit')
    axes[0].fill_between(range(len(backtest_account_values)), 
                         backtest_account_values[0], 
                         backtest_account_values,
                         where=[v < backtest_account_values[0] for v in backtest_account_values],
                         alpha=0.3, color='red', label='Loss')
    axes[0].set_title(f'Backtest: Portfolio Value\n{BACKTEST_TEST_START} to {BACKTEST_TEST_END}', 
                      fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Time Steps (1-min intervals)')
    axes[0].set_ylabel('Portfolio Value ($)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # 2. Drawdown Over Time
    running_max = np.maximum.accumulate(backtest_account_values)
    drawdowns = (np.array(backtest_account_values) - running_max) / running_max * 100
    axes[1].fill_between(range(len(drawdowns)), drawdowns, 0, alpha=0.7, color='red')
    axes[1].axhline(y=backtest_metrics['max_drawdown']*100, color='darkred', 
                    linestyle='--', linewidth=2, label=f"Max DD: {backtest_metrics['max_drawdown']:.1%}")
    axes[1].set_title('Backtest: Drawdown', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Time Steps (1-min intervals)')
    axes[1].set_ylabel('Drawdown (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./backtest_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Backtest plot saved to backtest_results.png")

✅ Backtest plot saved to backtest_results.png


/tmp/ipykernel_354197/4033188507.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [25]:
# === COMPARE HPO METRICS vs BACKTEST ===
if len(agg_df) > 0:
    print("=" * 70)
    print("📈 VALIDATION: HPO Estimates vs Final Backtest")
    print("=" * 70)
    print()
    print(f"{'Metric':<25} {'HPO Estimate':<20} {'Backtest':<20} {'Difference':<15}")
    print("-" * 70)
    
    # Sharpe comparison
    hpo_sharpe = best_config['mean_sharpe']
    bt_sharpe = backtest_metrics['sharpe_ratio']
    diff_sharpe = bt_sharpe - hpo_sharpe
    print(f"{'Sharpe Ratio':<25} {hpo_sharpe:<20.3f} {bt_sharpe:<20.3f} {diff_sharpe:+.3f}")
    
    # Return comparison
    hpo_return = best_config['mean_return']
    bt_return = backtest_metrics['total_return']
    diff_return = bt_return - hpo_return
    print(f"{'Total Return':<25} {hpo_return:<20.2%} {bt_return:<20.2%} {diff_return:+.2%}")
    
    # Drawdown comparison
    hpo_dd = best_config['worst_drawdown']
    bt_dd = backtest_metrics['max_drawdown']
    diff_dd = bt_dd - hpo_dd
    print(f"{'Max Drawdown':<25} {hpo_dd:<20.2%} {bt_dd:<20.2%} {diff_dd:+.2%}")
    
    print("-" * 70)
    
    # Verdict
    if bt_sharpe > 0.5 and bt_return > 0 and bt_dd > -0.15:
        print("\n✅ BACKTEST PASSED: Metrics look reasonable for deployment")
        print("   → Copy the BEST_PARAMS to the v2 trading notebook")
    elif bt_sharpe > 0 and bt_return > -0.05:
        print("\n⚠️ BACKTEST MARGINAL: Consider additional validation")
        print("   → Metrics are borderline - proceed with caution")
    else:
        print("\n❌ BACKTEST FAILED: Poor out-of-sample performance")
        print("   → Consider re-running HPO with different parameter ranges")
    
    print("\n" + "=" * 70)
    print("📋 FINAL RECOMMENDED PARAMS FOR v2 NOTEBOOK:")
    print("=" * 70)
    print(f'''
ERL_PARAMS = {{
    "learning_rate": {BEST_PARAMS['learning_rate']:.0e},
    "batch_size": {BEST_PARAMS['batch_size']},
    "gamma": {BEST_PARAMS['gamma']},
    "seed": 312,
    "net_dimension": {BEST_PARAMS['net_dimension']},
    "target_step": 5000,
    "eval_gap": 30,
    "eval_times": 1
}}
''')

📈 VALIDATION: HPO Estimates vs Final Backtest

Metric                    HPO Estimate         Backtest             Difference     
----------------------------------------------------------------------
Sharpe Ratio              5.523                1.997                -3.526
Total Return              5.82%                3.83%                -1.99%
Max Drawdown              -4.01%               -4.98%               -0.97%
----------------------------------------------------------------------

✅ BACKTEST PASSED: Metrics look reasonable for deployment
   → Copy the BEST_PARAMS to the v2 trading notebook

📋 FINAL RECOMMENDED PARAMS FOR v2 NOTEBOOK:

ERL_PARAMS = {
    "learning_rate": 3e-05,
    "batch_size": 1024,
    "gamma": 0.9,
    "seed": 312,
    "net_dimension": [32, 16],
    "target_step": 5000,
    "eval_gap": 30,
    "eval_times": 1
}



# $100K Tier HPO

Run the same walk-forward HPO at **$100K initial capital** with `max_stock=10` to find optimal hyperparameters for a smaller account.
Uses the same validation windows and parameter grid, but trains/tests with `ACCOUNT_CONFIG_100K`.

In [ ]:
# === $100K ACCOUNT CONFIGURATION ===
ACCOUNT_CONFIG_100K = {
    "initial_capital": 100_000,
    "max_stock": 1e2,
    "reward_scaling": 2**-11,
}

# Separate checkpoint for $100K HPO
CHECKPOINT_DIR_100K = './hpo_checkpoints_100k'
CHECKPOINT_FILE_100K = os.path.join(CHECKPOINT_DIR_100K, 'hpo_checkpoint.pkl')
RESULTS_BACKUP_FILE_100K = os.path.join(CHECKPOINT_DIR_100K, 'results_backup.csv')
os.makedirs(CHECKPOINT_DIR_100K, exist_ok=True)

print("=" * 60)
print("$100K TIER HPO CONFIGURATION")
print("=" * 60)
print(f"Initial Capital:  ${100_000:>12,.0f}")
print(f"Max Stock/Trade:  {1e2:>12.0f}")
print(f"Using same parameter grid: {len(param_combinations)} combinations")
print(f"Using same validation windows: {len(VALIDATION_WINDOWS)} windows")
print("=" * 60)

In [ ]:
# === $100K WALK-FORWARD GRID SEARCH ===
# Same structure as the $1M HPO but with $100K account config
# Checkpoints are stored separately in hpo_checkpoints_100k/

def save_checkpoint_100k(results, aggregated_results, completed_experiments):
    checkpoint = {
        'results': results,
        'aggregated_results': dict(aggregated_results),
        'completed_experiments': completed_experiments,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    with open(CHECKPOINT_FILE_100K, 'wb') as f:
        pickle.dump(checkpoint, f)
    if results:
        results_df = pd.DataFrame([r for r in results if 'error' not in r])
        if len(results_df) > 0:
            results_df.to_csv(RESULTS_BACKUP_FILE_100K, index=False)
    print(f"   💾 $100K Checkpoint saved ({len(completed_experiments)} experiments complete)")

def load_checkpoint_100k():
    if os.path.exists(CHECKPOINT_FILE_100K):
        try:
            with open(CHECKPOINT_FILE_100K, 'rb') as f:
                checkpoint = pickle.load(f)
            print(f"📂 $100K Checkpoint found from {checkpoint['timestamp']}")
            print(f"   Resuming from {len(checkpoint['completed_experiments'])} completed experiments")
            return checkpoint
        except Exception as e:
            print(f"⚠️ Failed to load checkpoint: {e}")
            return None
    return None

# Pre-flight connection check
print("🔌 Checking connection to Alpaca API...")
if not check_internet_connection():
    print("❌ Cannot connect to Alpaca API!")
    raise ConnectionError("No connection to Alpaca API. Please restore connection and re-run.")
print("✅ Connection OK\n")

checkpoint_100k = load_checkpoint_100k()

if checkpoint_100k:
    results_100k = checkpoint_100k['results']
    aggregated_results_100k = defaultdict(lambda: {
        'sharpe_ratios': [], 'returns': [], 'max_drawdowns': [], 'successful_windows': 0
    })
    for key, value in checkpoint_100k['aggregated_results'].items():
        aggregated_results_100k[key] = value
    completed_experiments_100k = set(checkpoint_100k['completed_experiments'])
    print(f"✅ Resuming $100K HPO - {len(completed_experiments_100k)} experiments already complete\n")
else:
    results_100k = []
    aggregated_results_100k = defaultdict(lambda: {
        'sharpe_ratios': [], 'returns': [], 'max_drawdowns': [], 'successful_windows': 0
    })
    completed_experiments_100k = set()
    print("🆕 Starting fresh $100K HPO run\n")

test_combinations_100k = param_combinations

total_experiments_100k = len(test_combinations_100k) * len(VALIDATION_WINDOWS)
remaining_100k = total_experiments_100k - len(completed_experiments_100k)

print(f"Testing {len(test_combinations_100k)} combinations across {len(VALIDATION_WINDOWS)} windows")
print(f"Total experiments: {total_experiments_100k}")
print(f"Already completed: {len(completed_experiments_100k)}")
print(f"Remaining: {remaining_100k}")

current_exp = 0
consecutive_failures = 0
graceful_exit = False

for lr, gamma, net_dim, bs in test_combinations_100k:
    if graceful_exit:
        break
    exp_key = f"lr{lr:.0e}_g{gamma}_net{'x'.join(map(str, net_dim))}_bs{bs}"
    
    for window_idx, (train_start, train_end, val_start, val_end) in enumerate(VALIDATION_WINDOWS):
        if graceful_exit:
            break
        current_exp += 1
        exp_id = f"{exp_key}_window{window_idx+1}"
        
        if exp_id in completed_experiments_100k:
            print(f"⏭️  Skipping {current_exp}/{total_experiments_100k} (already complete): {exp_id}")
            continue
        
        print(f"\n{'='*70}")
        print(f"$100K Experiment {current_exp}/{total_experiments_100k}")
        print(f"Config: LR={lr:.0e}, Gamma={gamma}, Net={net_dim}, Batch={bs}")
        print(f"Window {window_idx+1}: Train {train_start}→{train_end} | Val {val_start}→{val_end}")
        print(f"{'='*70}")
        
        exp_params = {
            "learning_rate": lr, "batch_size": bs, "gamma": gamma,
            "seed": 312, "net_dimension": net_dim,
            "target_step": 5000, "eval_gap": 30, "eval_times": 1
        }
        
        exp_cwd = f'./hpo_experiments_100k/{exp_key}/window_{window_idx+1}'
        os.makedirs(exp_cwd, exist_ok=True)
        
        try:
            train(
                start_date=train_start, end_date=train_end,
                ticker_list=ticker_list, data_source='alpaca', time_interval='1Min',
                technical_indicator_list=INDICATORS, drl_lib='elegantrl',
                env=env, model_name='ppo', if_vix=True,
                API_KEY=API_KEY, API_SECRET=API_SECRET, API_BASE_URL=API_BASE_URL,
                erl_params=exp_params, cwd=exp_cwd, break_step=1e5,
                account_config=ACCOUNT_CONFIG_100K
            )
            
            account_values = test(
                start_date=val_start, end_date=val_end,
                ticker_list=ticker_list, data_source='alpaca', time_interval='1Min',
                technical_indicator_list=INDICATORS, drl_lib='elegantrl',
                env=env, model_name='ppo', if_vix=True,
                API_KEY=API_KEY, API_SECRET=API_SECRET, API_BASE_URL=API_BASE_URL,
                cwd=exp_cwd, net_dimension=net_dim,
                account_config=ACCOUNT_CONFIG_100K
            )
            
            metrics = calculate_metrics(account_values)
            
            result = {
                'learning_rate': lr, 'gamma': gamma, 'net_dimension': net_dim,
                'batch_size': bs, 'window': window_idx + 1,
                'train_start': train_start, 'train_end': train_end,
                'val_start': val_start, 'val_end': val_end,
                'total_return': metrics['total_return'],
                'sharpe_ratio': metrics['sharpe_ratio'],
                'max_drawdown': metrics['max_drawdown'],
                'final_value': account_values[-1], 'exp_key': exp_key
            }
            results_100k.append(result)
            
            aggregated_results_100k[exp_key]['sharpe_ratios'].append(metrics['sharpe_ratio'])
            aggregated_results_100k[exp_key]['returns'].append(metrics['total_return'])
            aggregated_results_100k[exp_key]['max_drawdowns'].append(metrics['max_drawdown'])
            aggregated_results_100k[exp_key]['successful_windows'] += 1
            aggregated_results_100k[exp_key]['params'] = {
                'learning_rate': lr, 'gamma': gamma,
                'net_dimension': net_dim, 'batch_size': bs
            }
            
            print(f"\n✅ Window {window_idx+1} Results:")
            print(f"   Total Return: {metrics['total_return']:.2%}")
            print(f"   Sharpe Ratio: {metrics['sharpe_ratio']:.3f}")
            print(f"   Max Drawdown: {metrics['max_drawdown']:.2%}")
            
            completed_experiments_100k.add(exp_id)
            save_checkpoint_100k(results_100k, aggregated_results_100k, list(completed_experiments_100k))
            consecutive_failures = 0
            
        except Exception as e:
            error_msg = str(e)
            print(f"❌ Experiment failed: {error_msg}")
            
            if is_connection_error(error_msg):
                consecutive_failures += 1
                if not check_internet_connection() or consecutive_failures >= CONSECUTIVE_FAILURE_LIMIT:
                    print(f"\n🔌 CONNECTION LOST! Exiting gracefully.")
                    print(f"   Completed: {len(completed_experiments_100k)} experiments")
                    print(f"   ➡️  Re-run this cell after restoring connection.")
                    graceful_exit = True
                    break
            else:
                consecutive_failures = 0
                results_100k.append({
                    'learning_rate': lr, 'gamma': gamma, 'net_dimension': net_dim,
                    'batch_size': bs, 'window': window_idx + 1, 'error': error_msg,
                    'exp_key': exp_key
                })
                completed_experiments_100k.add(exp_id)
                save_checkpoint_100k(results_100k, aggregated_results_100k, list(completed_experiments_100k))

if not graceful_exit:
    print(f"\n{'='*70}")
    print(f"$100K Walk-forward grid search complete!")
    print(f"Successfully completed: {len([r for r in results_100k if 'error' not in r])}")
    print(f"Failed: {len([r for r in results_100k if 'error' in r])}")
    print(f"{'='*70}")

save_checkpoint_100k(results_100k, aggregated_results_100k, list(completed_experiments_100k))

In [ ]:
# === ANALYZE $100K HPO RESULTS ===
agg_summary_100k = []
for exp_key, data in aggregated_results_100k.items():
    if data['successful_windows'] == len(VALIDATION_WINDOWS):
        params = data['params']
        agg_summary_100k.append({
            'exp_key': exp_key,
            'learning_rate': params['learning_rate'],
            'gamma': params['gamma'],
            'net_dimension': str(params['net_dimension']),
            'batch_size': params['batch_size'],
            'mean_sharpe': np.mean(data['sharpe_ratios']),
            'std_sharpe': np.std(data['sharpe_ratios']),
            'min_sharpe': np.min(data['sharpe_ratios']),
            'mean_return': np.mean(data['returns']),
            'worst_drawdown': np.min(data['max_drawdowns']),
            'consistency': np.mean(data['sharpe_ratios']) / (np.std(data['sharpe_ratios']) + 0.1)
        })

agg_df_100k = pd.DataFrame(agg_summary_100k)

if len(agg_df_100k) > 0:
    print("=" * 80)
    print("$100K WALK-FORWARD AGGREGATED RESULTS")
    print(f"Configs that succeeded on all {len(VALIDATION_WINDOWS)} windows")
    print("=" * 80)
    
    print("\n📊 TOP 5 BY CONSISTENCY (mean_sharpe / std_sharpe):")
    print("-" * 80)
    top_consistent_100k = agg_df_100k.sort_values('consistency', ascending=False).head()
    print(top_consistent_100k[['learning_rate', 'gamma', 'net_dimension', 'batch_size',
                                'mean_sharpe', 'std_sharpe', 'consistency']].to_string())
    
    print("\n📊 TOP 5 BY MEAN SHARPE:")
    print("-" * 80)
    top_sharpe_100k = agg_df_100k.sort_values('mean_sharpe', ascending=False).head()
    print(top_sharpe_100k[['learning_rate', 'gamma', 'net_dimension', 'batch_size',
                           'mean_sharpe', 'mean_return', 'worst_drawdown']].to_string())
    
    best_100k = agg_df_100k.sort_values('consistency', ascending=False).iloc[0]
    print("\n" + "=" * 80)
    print("🏆 RECOMMENDED $100K CONFIGURATION (by consistency):")
    print("=" * 80)
    print(f"Learning Rate: {best_100k['learning_rate']:.0e}")
    print(f"Gamma: {best_100k['gamma']}")
    print(f"Network: {best_100k['net_dimension']}")
    print(f"Batch Size: {int(best_100k['batch_size'])}")
    print(f"\nAggregated Performance:")
    print(f"  Mean Sharpe: {best_100k['mean_sharpe']:.3f} ± {best_100k['std_sharpe']:.3f}")
    print(f"  Mean Return: {best_100k['mean_return']:.2%}")
    print(f"  Worst Drawdown: {best_100k['worst_drawdown']:.2%}")
    
    # Save results
    pd.DataFrame([r for r in results_100k if 'error' not in r]).to_csv(
        './hpo_per_window_results_100k.csv', index=False)
    agg_df_100k.to_csv('./hpo_aggregated_results_100k.csv', index=False)
    print("\n✅ Results saved to hpo_*_100k.csv")
    
    BEST_PARAMS_100K = {
        "learning_rate": float(best_100k['learning_rate']),
        "batch_size": int(best_100k['batch_size']),
        "gamma": float(best_100k['gamma']),
        "seed": 312,
        "net_dimension": eval(best_100k['net_dimension']),
        "target_step": 5000,
        "eval_gap": 30,
        "eval_times": 1
    }
    
    print(f"\n📋 COPY TO v2 NOTEBOOK FOR $100K DEPLOYMENT:")
    print(f'''
ERL_PARAMS = {{
    "learning_rate": {best_100k['learning_rate']:.0e},
    "batch_size": {int(best_100k['batch_size'])},
    "gamma": {best_100k['gamma']},
    "seed": 312,
    "net_dimension": {best_100k['net_dimension']},
    "target_step": 5000,
    "eval_gap": 30,
    "eval_times": 1
}}
''')
else:
    print("⚠️ No $100K configurations succeeded on all windows.")

In [ ]:
# === $100K BACKTEST ON HELD-OUT PERIOD ===
if len(agg_df_100k) > 0:
    backtest_cwd_100k = './backtest_final_100k'
    
    print("Training $100K model with best hyperparameters...")
    print(f"Period: {BACKTEST_TRAIN_START} to {BACKTEST_TRAIN_END}")
    
    train(
        start_date=BACKTEST_TRAIN_START, end_date=BACKTEST_TRAIN_END,
        ticker_list=ticker_list, data_source='alpaca', time_interval='1Min',
        technical_indicator_list=INDICATORS, drl_lib='elegantrl',
        env=env, model_name='ppo', if_vix=True,
        API_KEY=API_KEY, API_SECRET=API_SECRET, API_BASE_URL=API_BASE_URL,
        erl_params=BEST_PARAMS_100K, cwd=backtest_cwd_100k, break_step=2e5,
        account_config=ACCOUNT_CONFIG_100K
    )
    
    print("\nRunning $100K backtest on held-out period...")
    backtest_values_100k = test(
        start_date=BACKTEST_TEST_START, end_date=BACKTEST_TEST_END,
        ticker_list=ticker_list, data_source='alpaca', time_interval='1Min',
        technical_indicator_list=INDICATORS, drl_lib='elegantrl',
        env=env, model_name='ppo', if_vix=True,
        API_KEY=API_KEY, API_SECRET=API_SECRET, API_BASE_URL=API_BASE_URL,
        cwd=backtest_cwd_100k, net_dimension=BEST_PARAMS_100K['net_dimension'],
        account_config=ACCOUNT_CONFIG_100K
    )
    
    backtest_metrics_100k = calculate_metrics(backtest_values_100k)
    
    print("\n" + "=" * 70)
    print("📊 $100K BACKTEST RESULTS (Held-Out Period)")
    print("=" * 70)
    print(f"Period: {BACKTEST_TEST_START} to {BACKTEST_TEST_END}")
    print(f"  Initial Value:   ${backtest_values_100k[0]:>12,.2f}")
    print(f"  Final Value:     ${backtest_values_100k[-1]:>12,.2f}")
    print(f"  Total Return:    {backtest_metrics_100k['total_return']:>10.2%}")
    print(f"  Sharpe Ratio:    {backtest_metrics_100k['sharpe_ratio']:>10.3f}")
    print(f"  Max Drawdown:    {backtest_metrics_100k['max_drawdown']:>10.2%}")
    
    # Compare with $1M backtest if available
    try:
        print("\n" + "=" * 70)
        print("📈 $1M vs $100K BACKTEST COMPARISON")
        print("=" * 70)
        print(f"{'Metric':<25} {'$1M':<20} {'$100K':<20}")
        print("-" * 65)
        print(f"{'Total Return':<25} {backtest_metrics['total_return']:<20.2%} {backtest_metrics_100k['total_return']:<20.2%}")
        print(f"{'Sharpe Ratio':<25} {backtest_metrics['sharpe_ratio']:<20.3f} {backtest_metrics_100k['sharpe_ratio']:<20.3f}")
        print(f"{'Max Drawdown':<25} {backtest_metrics['max_drawdown']:<20.2%} {backtest_metrics_100k['max_drawdown']:<20.2%}")
        print("=" * 65)
    except NameError:
        print("\n(Run the $1M backtest section first to compare)")
else:
    print("⚠️ No $100K HPO results available. Run the $100K grid search first.")